In [28]:
import os
import random
from datetime import datetime, timedelta

import pandas as pd
from faker import Faker


fake = Faker()
random.seed(42)
Faker.seed(42)

NUM_CUSTOMERS = 600
NUM_PRODUCTS = 600
NUM_ORDERS = 1000
NUM_ORDER_ITEMS = 3000

RAW_DATA_PATH = "../data/raw"

os.makedirs(RAW_DATA_PATH, exist_ok=True)

In [29]:
# Master data

CUSTOMER_TYPES = [
    "REGULAR",
    "PREMIUM",
    "VIP"
]

ORDER_STATUS = [
    "PLACED",
    "SHIPPED",
    "DELIVERED",
    "CANCELLED",
    "RETURNED"
]

REGIONS = [
    "NORTH",
    "SOUTH",
    "EAST",
    "WEST"
]

PRODUCT_CATEGORIES = {
    "Electronics": [
        "Mobile",
        "Laptop",
        "Accessories"
    ],

    "Clothing": [
        "Men",
        "Women",
        "Kids"
    ],

    "Home": [
        "Kitchen",
        "Furniture",
        "Decor"
    ],

    "Books": [
        "Fiction",
        "Education",
        "Biography"
    ]
}

In [30]:

# Helper Functions


def random_registration_date():
    """
    Generate a random registration date
    within the last 3 years.
    """

    start_date = datetime.now() - timedelta(days=3 * 365)

    random_days = random.randint(0, 3 * 365)

    return (
        start_date + timedelta(days=random_days)
    ).strftime("%Y-%m-%d")


def random_order_datetime():
    """
    Generate a random order datetime
    within the last 2 years.
    """

    start_date = datetime.now() - timedelta(days=730)

    random_seconds = random.randint(
        0,
        730 * 24 * 60 * 60
    )

    return start_date + timedelta(
        seconds=random_seconds
    )

In [31]:
# Customers

customers = []

for customer_id in range(1, NUM_CUSTOMERS + 1):

    name = fake.name()

    email = fake.email()

    customer_type = random.choice(CUSTOMER_TYPES)

    registration_date = random_registration_date()

    customers.append({

        "customer_id": customer_id,

        "customer_name": name,

        "email": email,

        "registration_date": registration_date,

        "customer_type": customer_type

    })

In [32]:
# Introduce Invalid Emails


invalid_email_count = int(
    NUM_CUSTOMERS * 0.02
)

indices = random.sample(
    range(NUM_CUSTOMERS),
    invalid_email_count
)

for i in indices:

    email = customers[i]["email"]

    choice = random.randint(1, 3)

    if choice == 1:

        customers[i]["email"] = email.replace("@", "")

    elif choice == 2:

        customers[i]["email"] = email.split("@")[0]

    else:

        customers[i]["email"] = email.replace(".com", "")

In [33]:
customers_df = pd.DataFrame(customers)

customers_df.to_csv(
    os.path.join(
        RAW_DATA_PATH,
        "customers.csv"
    ),
    index=False
)

print("customers.csv generated successfully.")

customers.csv generated successfully.


In [34]:
# Product Master Data

PRODUCTS = {
    "Electronics": {
        "Mobile": [
            "iPhone 15", "Samsung Galaxy S24", "OnePlus 12",
            "Google Pixel 8", "Realme GT", "Redmi Note 13"
        ],
        "Laptop": [
            "MacBook Air", "Dell Inspiron", "HP Pavilion",
            "Lenovo ThinkPad", "Asus VivoBook", "Acer Aspire"
        ],
        "Accessories": [
            "Bluetooth Speaker", "Wireless Mouse", "Keyboard",
            "Power Bank", "USB Cable", "Smart Watch"
        ]
    },

    "Clothing": {
        "Men": [
            "Cotton Shirt", "Formal Pant", "T-Shirt",
            "Jeans", "Jacket", "Hoodie"
        ],
        "Women": [
            "Kurti", "Saree", "Top",
            "Jeans", "Dress", "Sweater"
        ],
        "Kids": [
            "Kids T-Shirt", "School Uniform", "Shorts",
            "Jacket", "Cap", "Shoes"
        ]
    },

    "Home": {
        "Kitchen": [
            "Mixer Grinder", "Pressure Cooker", "Gas Stove",
            "Knife Set", "Dinner Set", "Water Bottle"
        ],
        "Furniture": [
            "Office Chair", "Study Table", "Sofa",
            "Wardrobe", "Bookshelf", "Bed"
        ],
        "Decor": [
            "Wall Clock", "Flower Vase", "Painting",
            "Lamp", "Mirror", "Carpet"
        ]
    },

    "Books": {
        "Education": [
            "Python Programming", "DBMS", "Operating System",
            "Data Structures", "Machine Learning", "Java Programming"
        ],
        "Fiction": [
            "The Alchemist", "Harry Potter", "Sherlock Holmes",
            "The Hobbit", "1984", "The Silent Patient"
        ],
        "Biography": [
            "Steve Jobs", "APJ Abdul Kalam",
            "Elon Musk", "Wings of Fire",
            "Becoming", "Long Walk to Freedom"
        ]
    }
}

In [35]:
products = []

for product_id in range(1, NUM_PRODUCTS + 1):

    category = random.choice(list(PRODUCTS.keys()))

    subcategory = random.choice(list(PRODUCTS[category].keys()))

    product_name = random.choice(PRODUCTS[category][subcategory])

    cost_price = round(random.uniform(100, 50000), 2)

    products.append({
        "product_id": product_id,
        "product_name": product_name,
        "category": category,
        "subcategory": subcategory,
        "cost_price": cost_price
    })

In [36]:
# Mixed Case + Extra Spaces

issue_count = int(NUM_PRODUCTS * 0.05)

indices = random.sample(range(NUM_PRODUCTS), issue_count)

for i in indices:

    choice = random.randint(1, 4)

    name = products[i]["product_name"]

    if choice == 1:
        products[i]["product_name"] = name.upper()

    elif choice == 2:
        products[i]["product_name"] = name.lower()

    elif choice == 3:
        products[i]["product_name"] = "   " + name

    else:
        products[i]["product_name"] = name + "   "

In [37]:
products_df = pd.DataFrame(products)

products_df.to_csv(
    os.path.join(RAW_DATA_PATH, "products.csv"),
    index=False
)

print("products.csv generated successfully!")

products_df.head()

products.csv generated successfully!


,product_id,product_name,category,subcategory,cost_price
0,1,OnePlus 12,Electronics,Mobile,49796.78
1,2,SAMSUNG GALAXY S24,Electronics,Mobile,44374.24
2,3,Jeans,Clothing,Women,18613.47
3,4,Long Walk to Freedom,Books,Biography,36357.30
4,5,Steve Jobs,Books,Biography,41701.28


In [38]:

# Generate Orders

orders = []

for order_id in range(1, NUM_ORDERS + 1):

    customer_id = random.randint(1, NUM_CUSTOMERS)

    status = random.choice(ORDER_STATUS)

    region = random.choice(REGIONS)

    order_date = random_order_datetime()

    orders.append({
        "order_id": order_id,
        "customer_id": customer_id,
        "region_code": region,
        "status": status,
        "order_date": order_date.strftime("%Y-%m-%d %H:%M:%S")
    })

In [39]:


missing_count = int(NUM_ORDERS * 0.05)

indices = random.sample(range(NUM_ORDERS), missing_count)

for i in indices:
    orders[i]["customer_id"] = None

In [40]:
# wrong date formats

wrong_date_count = int(NUM_ORDERS * 0.05)

indices = random.sample(range(NUM_ORDERS), wrong_date_count)

for i in indices:

    dt = datetime.strptime(
        orders[i]["order_date"],
        "%Y-%m-%d %H:%M:%S"
    )

    orders[i]["order_date"] = dt.strftime("%d-%m-%Y")

In [41]:
orders_df = pd.DataFrame(orders)

orders_df.to_csv(
    os.path.join(RAW_DATA_PATH, "orders.csv"),
    index=False
)

print("orders.csv generated successfully!")

orders_df.head()

orders.csv generated successfully!


,order_id,customer_id,region_code,status,order_date
0,1,77.0,NORTH,RETURNED,2025-04-25 02:28:13
1,2,495.0,WEST,PLACED,2025-12-26 07:15:28
2,3,169.0,WEST,RETURNED,2024-12-05 15:14:30
3,4,55.0,EAST,PLACED,2025-01-12 06:21:12
4,5,195.0,EAST,CANCELLED,2025-12-31 17:03:08


In [55]:
# -----------------------------
# Generate Order Items
# -----------------------------

order_items = []

for item_id in range(1, NUM_ORDER_ITEMS + 1):

    order_id = random.randint(1, NUM_ORDERS)

    product_id = random.randint(1, NUM_PRODUCTS)

    quantity = random.randint(1, 5)

    # Fetch product cost price
    cost_price = products_df.loc[
        products_df["product_id"] == product_id,
        "cost_price"
    ].values[0]

    # Selling price = 10% to 50% higher than cost price
    unit_price = round(
        cost_price * random.uniform(1.10, 1.50),
        2
    )

    discount_percent = random.randint(0, 50)

    order_items.append({
        "item_id": item_id,
        "order_id": order_id,
        "product_id": product_id,
        "quantity": quantity,
        "unit_price": unit_price,
        "discount_percent": discount_percent
    })

In [56]:
# Negative Quantities (Returns)


negative_count = int(NUM_ORDER_ITEMS * 0.03)

indices = random.sample(
    range(NUM_ORDER_ITEMS),
    negative_count
)

for i in indices:

    order_items[i]["quantity"] = -abs(
        order_items[i]["quantity"]
    )

In [57]:
order_items_df = pd.DataFrame(order_items)

order_items_df.to_csv(
    os.path.join(
        RAW_DATA_PATH,
        "order_items.csv"
    ),
    index=False
)

print("order_items.csv generated successfully!")

order_items_df.head()

order_items.csv generated successfully!


,item_id,order_id,product_id,quantity,unit_price,discount_percent
0,1,250,441,5,31077.59,34
1,2,581,465,4,25399.47,33
2,3,486,221,1,23470.46,31
3,4,915,496,2,34395.38,38
4,5,789,190,2,40535.86,24


In [58]:
print("=" * 50)
print("Data Generation Summary")
print("=" * 50)

print(f"Customers     : {len(customers_df)}")
print(f"Products      : {len(products_df)}")
print(f"Orders        : {len(orders_df)}")
print(f"Order Items   : {len(order_items_df)}")

print("\nMissing Customer IDs :", orders_df["customer_id"].isna().sum())

print("Negative Quantities :", (order_items_df["quantity"] < 0).sum())

print("Invalid Emails :", customers_df["email"].str.contains("@").sum() != len(customers_df))

Data Generation Summary
Customers     : 600
Products      : 600
Orders        : 1000
Order Items   : 3000

Missing Customer IDs : 50
Negative Quantities : 90
Invalid Emails : True


## **Data Cleaning**

In [59]:
import pandas as pd


orders_df = pd.read_csv("../data/raw/orders.csv")

products_df = pd.read_csv("../data/raw/products.csv")

customers_df = pd.read_csv("../data/raw/customers.csv")

order_items_df = pd.read_csv("../data/raw/order_items.csv")

print("All raw datasets loaded successfully.")

All raw datasets loaded successfully.


In [60]:
# -----------------------------
# Cleaning Report
# -----------------------------

issues_report = {
    "missing_customer_ids": 0,
    "invalid_dates": 0,
    "invalid_emails": 0,
    "negative_quantities": 0,
    "invalid_order_references": 0,
    "product_name_changes": 0
}

In [61]:
from datetime import datetime
import pandas as pd

def clean_orders(df):

    df = df.copy()

    fixed_dates = []

    invalid_dates = 0

    for value in df["order_date"]:

        try:
            dt = datetime.strptime(
                str(value),
                "%Y-%m-%d %H:%M:%S"
            )

        except:

            try:

                dt = datetime.strptime(
                    str(value),
                    "%d-%m-%Y"
                )

            except:

                dt = pd.NaT
                invalid_dates += 1

        fixed_dates.append(dt)

    df["order_date"] = fixed_dates

    issues_report["invalid_dates"] = invalid_dates

    missing = df["customer_id"].isna().sum()

    issues_report["missing_customer_ids"] = missing

    df["customer_id"] = df["customer_id"].fillna(-1).astype(int)

    return df

In [62]:
orders_clean = clean_orders(orders_df)

orders_clean.head()

,order_id,customer_id,region_code,status,order_date
0,1,77,NORTH,RETURNED,2025-04-25 02:28:13
1,2,495,WEST,PLACED,2025-12-26 07:15:28
2,3,169,WEST,RETURNED,2024-12-05 15:14:30
3,4,55,EAST,PLACED,2025-01-12 06:21:12
4,5,195,EAST,CANCELLED,2025-12-31 17:03:08


In [63]:
print("Missing Customer IDs :", issues_report["missing_customer_ids"])

print("Invalid Dates :", issues_report["invalid_dates"])

orders_clean.info()

Missing Customer IDs : 50
Invalid Dates : 0
<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   order_id     1000 non-null   int64         
 1   customer_id  1000 non-null   int64         
 2   region_code  1000 non-null   str           
 3   status       1000 non-null   str           
 4   order_date   1000 non-null   datetime64[us]
dtypes: datetime64[us](1), int64(2), str(2)
memory usage: 39.2 KB


In [64]:
# -----------------------------
# Clean Product Names
# -----------------------------

def clean_products(df):

    df = df.copy()

    changes = 0

    cleaned_names = []

    for name in df["product_name"]:

        original_name = str(name)

        # Remove leading/trailing spaces
        cleaned = original_name.strip()

        # Remove multiple spaces
        cleaned = " ".join(cleaned.split())

        # Convert to Title Case
        cleaned = cleaned.title()

        if cleaned != original_name:
            changes += 1

        cleaned_names.append(cleaned)

    df["product_name"] = cleaned_names

    issues_report["product_name_changes"] = changes

    return df

In [65]:
products_clean = clean_products(products_df)

products_clean.head(10)

,product_id,product_name,category,subcategory,cost_price
0,1,Oneplus 12,Electronics,Mobile,49796.78
1,2,Samsung Galaxy S24,Electronics,Mobile,44374.24
2,3,Jeans,Clothing,Women,18613.47
3,4,Long Walk To Freedom,Books,Biography,36357.30
4,5,Steve Jobs,Books,Biography,41701.28
5,6,The Alchemist,Books,Fiction,34545.01
6,7,Jeans,Clothing,Women,47532.70
7,8,Water Bottle,Home,Kitchen,18430.01
8,9,Knife Set,Home,Kitchen,13866.48
9,10,Macbook Air,Electronics,Laptop,33172.16


In [66]:
print("Product Names Modified :", issues_report["product_name_changes"])

Product Names Modified : 136


In [68]:
products_clean.to_csv(
    "data/cleaned/products_clean.csv",
    index=False
)